# Chapter 2 — What a closure is

Step 2 builds **phase A**: the reflection that turns a just-defined Python
function into a `Handle` — the thesis's `(code identity, typed environment,
environment values)` made concrete. Phase A runs on *every closure rebuild*,
i.e. every iteration of your hot loop, so its contract is strict: read the
code object, read the cells, summarize types via memoized fingerprints —
**no parse, no IR, no compile, ever** — and never fail on missing source
(that's phase B's loud error, ch07). It *does* fail loudly on an untypeable
capture: better at the def site than inside a cache key.

| File | Counted lines / cap | What |
|---|---|---|
| `src/pdum/dsl/kernel/capture.py` | 67 / 85 | `make_handle`, `Handle`, `SourceSnapshot`, the Handle ValueKind |
| `src/pdum/dsl/kernel/api.py` | 8 / 50 | `@jit` (capture only; the call path arrives in ch09) |

Glossary terms settled this chapter: *Handle, SourceSnapshot* — see
[GLOSSARY.md](GLOSSARY.md).

In [1]:
from pdum.dsl.kernel.api import jit
from pdum.dsl.kernel.capture import make_handle


def make_closure(x):
    @jit(kind="device")
    def inner(y):
        return x + y

    return inner


f5, f6 = make_closure(5), make_closure(6)
print("a Handle:            ", f5)
print("identities equal:    ", f5.fntype == f6.fntype, "  <- values differ, ONE specialization")
print("envs (runtime data): ", f5.env, "|", f6.env)
print("fingerprints equal:  ", f5.fp == f6.fp)
print("float capture:       ", make_closure(3.0).fntype == f5.fntype, " <- type changed: its own identity")

a Handle:             Handle[device](Fn<make_closure.<locals>.inner>(i64), env=['x'])
identities equal:     True   <- values differ, ONE specialization
envs (runtime data):  {'x': 5} | {'x': 6}
fingerprints equal:   True
float capture:        False  <- type changed: its own identity


What `@jit` actually did: looked up the memoized `SourceSnapshot` for the code
object (taken once, at first decoration, while `linecache` is coherent),
read the closure cells in `co_freevars` order, fingerprinted each capture,
and looked up the memoized `FnType`. Everything after the first decoration
of a given code object is dictionary probes. Let's price it:

In [2]:
import timeit

n = 50_000
t = min(timeit.repeat(lambda: make_closure(5), number=n, repeat=5)) / n * 1e9
print(f"phase A, memo-warm: {t:6.0f} ns per closure rebuild")
# Rebuilding a 300-module network every training step at this price is
# sub-millisecond — the claim from the NN detour, now measurable.

phase A, memo-warm:   1325 ns per closure rebuild


In [3]:
def make_shader(center):
    @jit(kind="fragment")
    def shader():
        return center

    return shader


h = make_shader((320.0, 240.0))
print(h.fntype, "  <- a Tuple inside the identity: M0's top gap, closed at capture")

# CAREFUL: only enclosing-FUNCTION names become closure cells. A module- or
# cell-level name is a GLOBAL — phase A does not capture it:
center = (320.0, 240.0)


@jit(kind="fragment")
def shader_global():
    return center


print("global 'center' captured?", shader_global.freevars, shader_global.fntype)
# Globals are a different mechanism: classify_names decides their fate at
# lowering (ch07), and per-call guards defend against drift (ch09).

Fn<make_shader.<locals>.shader>((f64, f64))   <- a Tuple inside the identity: M0's top gap, closed at capture
global 'center' captured? () Fn<shader_global>()


In [4]:
# Live-coding invalidation, now through real Handles: an unchanged re-run
# produces a VALUE-EQUAL code object (hit); a one-token edit misses.
SRC = "def f(k):\n    def g():\n        return k\n    return g\n"


def build(source):
    ns = {}
    exec(compile(source, "<cell>", "exec"), ns)
    return make_handle(ns["f"](7), "device")


print("re-run unchanged:", build(SRC).fntype == build(SRC).fntype)
print("after an edit:   ", build(SRC).fntype == build(SRC.replace("return k", "return k + 1")).fntype)

re-run unchanged: True
after an edit:    False


## The program is the parameter container

Composition is structural: a capture that is itself a `Handle` contributes
its `FnType` to the parent's `env_types`. Build a network out of small
closures each owning its own weights, rebuild the whole object graph every
step with fresh values — the root identity never moves. (Neither JAX nor
PyTorch can offer this: one retraces on rebuilt closures, the other needs
stateful modules. See the desiderata §2.1 note.)

In [5]:
def dense(w, b):
    @jit()
    def layer(x):
        return w * x + b

    return layer


def net(w1, b1, w2, b2):
    inner = dense(w1, b1)

    @jit()
    def outer(x):
        return inner(x) * w2 + b2

    return outer


a = net(1.0, 2.0, 3.0, 4.0)
print("composed identity: ", a.fntype)
print("env_types:         ", a.env_types, "  <- the child's FnType is INSIDE the identity")
print("new weights, hit?  ", net(0.9, 1.9, 2.9, 3.9).fntype == a.fntype)

composed identity:  Fn<net.<locals>.outer>(f64, Fn<dense.<locals>.layer>(f64, f64), f64)
env_types:          (f64, Fn<dense.<locals>.layer>(f64, f64), f64)   <- the child's FnType is INSIDE the identity
new weights, hit?   True


In [6]:
from pdum.dsl.kernel.valuekind import typeof

# Handles are first-class values of the type system itself:
h = make_closure(5)
print(typeof(h))
print(typeof((h, make_closure(6))), "  <- a tuple of layers: identities compose")

Fn<make_closure.<locals>.inner>(i64)
(Fn<make_closure.<locals>.inner>(i64), Fn<make_closure.<locals>.inner>(i64))   <- a tuple of layers: identities compose


## Pricing the compositional model: a 100-block mock transformer

The claim from the NN detour (desiderata §2.1, working notes in
`design/deep-learning-notes.md`): identity is built **bottom-up**, each
Handle carrying the precomputed digest of its subtree, so a parent
incorporates a child in O(1) — no re-traversal. Two consequences to
*measure*, not assert: per-step rebuild cost scales with module count, and an
**unchanged subtree** (a frozen backbone) can be reused across steps
essentially for free. The "attention" below is a mock — no tensor ops exist
yet; what we are pricing is pure identity construction (phase A), which is
exactly what a training loop pays per step before any math runs.

Watch the **ns-per-handle column** in the output: fingerprint *construction*
is O(1) per child, but Python re-hashes nested fingerprint tuples on dict
probes (tuples don't memoize their hash), so deep spines drift mildly
superlinear. That drift is a known, pre-shaped escalation — `Handle.fp`
becomes a flat memoized digest (the `Node.key` technique) if the step-9
microbench gate ever demands it — and the frozen-backbone pattern below makes
it moot for the common fine-tuning case anyway.

In [7]:
import timeit


def attention(wq, wk, wv, wo):
    @jit()
    def attn(x):
        return wq * x + wk * x + wv * x + wo  # mocked; real tensor ops arrive much later

    return attn


def mlp(w1, w2):
    @jit()
    def ff(x):
        return w2 * (w1 * x)

    return ff


def block(p):
    a, m = attention(p, p, p, p), mlp(p, p)

    @jit()
    def blk(x):
        return m(a(x))

    return blk


def compose(f, g):
    @jit()
    def seq(x):
        return g(f(x))

    return seq


def transformer(weights):
    model = block(weights[0])
    for p in weights[1:]:
        model = compose(model, block(p))
    return model


weights = [float(i) for i in range(100)]
model = transformer(weights)
print("root identity (truncated):", repr(model.fntype)[:90], "...")
print("rebuilt with new weights, same identity:",
      transformer([w + 0.5 for w in weights]).fntype == model.fntype)
print()
for n in (10, 50, 100):
    handles = 4 * n - 1  # attn + ff + blk per block, plus n-1 composes
    t = min(timeit.repeat(lambda: transformer(weights[:n]), number=5, repeat=3)) / 5
    print(f"{n:4d} blocks: {t * 1e6:7.0f} µs per full rebuild   ({t / handles * 1e9:5.0f} ns per handle)")

root identity (truncated): Fn<compose.<locals>.seq>(Fn<compose.<locals>.seq>(Fn<compose.<locals>.seq>(Fn<compose.<loc ...
rebuilt with new weights, same identity: True

  10 blocks:      73 µs per full rebuild   ( 1872 ns per handle)
  50 blocks:     473 µs per full rebuild   ( 2378 ns per handle)
 100 blocks:    1227 µs per full rebuild   ( 3076 ns per handle)


In [8]:
# The unchanged-subtree trick: a frozen backbone is built ONCE, outside the
# loop; each "training step" rebuilds only the trainable head and one compose.
# Identity is bit-identical either way — phase A cost collapses to the spine
# that actually changed. (Fine-tuning and adapters fall out of the model.)
backbone = transformer(weights[:99])


def step_full_rebuild():
    return compose(transformer(weights[:99]), block(weights[99]))


def step_frozen_backbone():
    return compose(backbone, block(weights[99]))


assert step_full_rebuild().fntype == step_frozen_backbone().fntype

t_full = min(timeit.repeat(step_full_rebuild, number=5, repeat=3)) / 5
t_frozen = min(timeit.repeat(step_frozen_backbone, number=5, repeat=3)) / 5
print(f"rebuild all 100 blocks: {t_full * 1e6:7.0f} µs/step")
print(f"frozen backbone:        {t_frozen * 1e6:7.0f} µs/step   ({t_full / t_frozen:4.0f}x cheaper, same identity)")

rebuild all 100 blocks:    1231 µs/step
frozen backbone:             18 µs/step   (  69x cheaper, same identity)


In [9]:
# Two edge cases phase A must survive:

# 1. A self-referential closure's own cell is EMPTY at decoration time.
captured = {}


def grab(fn):
    captured["h"] = make_handle(fn, "device")
    return fn


def factory():
    @grab
    def rec(n):
        return 1 if n == 0 else rec(n - 1)

    return rec


factory()
h = captured["h"]
print("self-cell skipped:", h.env, "| template still knows the name:", h.freevars)

# 2. Source unavailable (some REPLs): phase A succeeds; the snapshot is None
#    and phase B (ch07) raises the loud NoSourceError if lowering is attempted.
ns = {}
exec(compile(SRC, "<no-file>", "exec"), ns)
print("no source -> snapshot:", make_handle(ns["f"](1), "device").snapshot)

self-cell skipped: {} | template still knows the name: ('rec',)
no source -> snapshot: None


In [10]:
# The snapshot: decoration-time source, memoized per code object. Taking it
# EAGERLY is the first half of the stale-source defense (a later on-disk edit
# cannot retroactively change what we captured); phase B's coherence check
# (ch07) is the second half.
snap = f5.snapshot
print(snap.qualname, "@", snap.filename, "line", snap.firstlineno)
print(snap.text)
print("memoized:", make_closure(1).snapshot is make_closure(2).snapshot)

make_closure.<locals>.inner @ /tmp/ipykernel_32693/3405070283.py line 6
@jit(kind="device")
def inner(y):
    return x + y

memoized: True


## What we can't do yet

- **Calling a Handle does nothing yet** — there is no cache to dispatch into
  (ch03) and no backend to run on (ch09). A Handle is pure identity + data.
- Lowering the body (`snap.text` → typed IR) is ch07; the snapshot coherence
  check lives there.
- The `FnType` memo is unbounded — eviction policy arrives with the cache
  tier (ch03), per the hazard doc's L-cache warning.

**Budget after step 2:** kernel 230 / 1150 counted lines.

**Next:** ch03 — one compile per signature: the two-tier cache, proven with
dummy artifacts before any compiler exists.